# CS224N L01 — Softmax Probability Computation

**Waypoint**: WP03 — Word2Vec / Skip-gram / Softmax  
**Concept**: Softmax probability computation  
**Anchor**: slides p28-p30 (`P(o|c) = exp(u_o^T v_c) / Σ_w exp(u_w^T v_c)`); notes §3.2 Eq.4-Eq.5

This capsule demonstrates the full softmax over a tiny vocabulary step-by-step:
1. Center word vector v_c and outside word vectors u_w
2. Dot product scores u_w^T v_c for each candidate outside word
3. Exponentiation exp(score)
4. Normalization (divide by partition function Z)
5. Final probability distribution P(o|c)

**Why code?** The denominator over the whole vocabulary is easy to gloss over in formulas; running it on a tiny vocabulary shows dot product scores, exponentials, normalization, and probability mass.

In [ ]:
import math
import json

# Tiny vocabulary
vocab = ["bank", "river", "money", "finance", "stream"]
V = len(vocab)       # vocabulary size = 5
d = 3                # embedding dimension (tiny for transparency)

# Center word vectors v (V x d)
v = [
    [ 0.50, -0.14,  0.65],   # bank
    [ 0.14,  0.76,  0.24],   # river
    [ 0.44,  0.13,  0.42],   # money
    [-0.55,  0.34,  0.28],   # finance
    [ 0.81, -0.40,  0.27],   # stream
]

# Outside word vectors u (V x d)
u = [
    [ 0.06,  0.31, -0.14],   # bank
    [ 0.64, -0.23,  0.18],   # river
    [-0.10,  0.31, -0.81],   # money
    [ 0.30, -0.15,  0.45],   # finance
    [ 0.65,  0.26, -0.46],   # stream
]

def dot_product(a, b):
    return sum(ai * bi for ai, bi in zip(a, b))

print(f"Vocabulary (|V|={V}): {vocab}")
print(f"Embedding dimension (d): {d}")
print(f"\nCenter vectors v:")
for i, w in enumerate(vocab):
    print(f"  v_{w:<8s} = {v[i]}")
print(f"\nOutside vectors u:")
for i, w in enumerate(vocab):
    print(f"  u_{w:<8s} = {u[i]}")

## Step 1: Pick a center word

We choose **'bank'** as the center word. Its center vector v_c will be dotted with every outside vector u_w.

In [ ]:
center_idx = 0   # 'bank'
center_word = vocab[center_idx]
v_c = v[center_idx]

print(f"Center word = '{center_word}', v_c = {v_c}\n")
print("Step 2: Dot product scores  u_w^T v_c")
scores = []
for i, w in enumerate(vocab):
    score = dot_product(u[i], v_c)
    scores.append(score)
    print(f"  u_{w:<8s}^T v_c = {score:.4f}")
print(f"\nScore vector: {[round(s, 4) for s in scores]}")

In [ ]:
# Step 3: Exponentiate
exp_scores = [math.exp(s) for s in scores]
print("Step 3: Exponentiate  exp(score)")
for i, w in enumerate(vocab):
    print(f"  exp({scores[i]:+.4f}) = {exp_scores[i]:.4f}")

# Step 4: Partition function Z
Z = sum(exp_scores)
print(f"\nStep 4: Partition function Z = {Z:.4f}")

# Step 5: Softmax probabilities
probs = [e / Z for e in exp_scores]
print(f"\nStep 5: Softmax probabilities  P(o|c) = exp(score) / Z")
for i, w in enumerate(vocab):
    print(f"  P({w:<8s} | {center_word}) = {probs[i]:.4f}")
print(f"\nSum check: {sum(probs):.6f}")

In [ ]:
# Summary table
print(f"{'Outside word':<14s} {'dot score':>10s} {'exp(score)':>12s} {'P(o|c)':>10s}")
print(f"{'─' * 14} {'─' * 10} {'─' * 12} {'─' * 10}")
for i, w in enumerate(vocab):
    print(f"{w:<14s} {scores[i]:>10.4f} {exp_scores[i]:>12.4f} {probs[i]:>10.4f}")
print(f"{'─' * 14} {'─' * 10} {'─' * 12} {'─' * 10}")
print(f"{'SUM':<14s} {'':>10s} {Z:>12.4f} {sum(probs):>10.4f}")

In [ ]:
import matplotlib.pyplot as plt

max_prob_idx = probs.index(max(probs))
colors = ["#e74c3c" if i == max_prob_idx else "#3498db" for i in range(V)]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(vocab, probs, color=colors, edgecolor="black", linewidth=0.5)

for bar, prob in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{prob:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_xlabel("Outside word (o)", fontsize=12)
ax.set_ylabel("P(o | c='bank')", fontsize=12)
ax.set_title("Softmax Probability Distribution P(o | c='bank')\nCS224N L01 Word Vectors — Code Capsule",
             fontsize=13, fontweight="bold")
ax.set_ylim(0, max(probs) * 1.25)
ax.axhline(y=1.0/V, color="gray", linestyle="--", alpha=0.5, label=f"Uniform (1/|V|={1/V:.3f})")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Key Insight

- **'river'** gets the highest probability because its outside vector has the largest dot product with v_bank.
- The denominator **Z** requires summing over **ALL |V| words**.
- In real word2vec, |V| ~ 10,000-100,000, making Z very expensive to compute.
- This is why **negative sampling** (WP04) replaces the full softmax with a binary classification objective.

---
*Related concepts:* [[Lectures/L01-word-vectors/00-concept-glossary#softmax|softmax]], [[Lectures/L01-word-vectors/00-concept-glossary#dot-product|dot product]], [[Lectures/L01-word-vectors/00-concept-glossary#softmax-probability-computation|softmax probability computation]]